* Создали 300 синтетических скважин и учит модель прогнозировать начальный дебит нефти ```qi_bopd```

* Установили библиотеки 1 раз

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

In [61]:
#Создание синтетического набора по 300 скважтнам 
# Фиксированное зерно гененратона обеспечение воспроизводимость результата

rng= np.random. default_rng(42)
n_wells= 300

porosity= rng.uniform(0.08,0.28, n_wells) 
permeability_md= rng.lognormal(
    mean=np.log(80), sigma=0.7, size=n_wells
).clip(1,500)
permeability_md

net_pay_thickness_ft= rng.uniform(10,100, n_wells)

boi_rb_stb= rng.uniform(1.10,1.50, n_wells)

choke_64in= rng.uniform(16,64, n_wells)

water_cut= rng.uniform(0.02, 0.7, n_wells)

## Необъяснённая геологическая и операционная изменчивость
noise= rng.normal(0, n_wells)

#Ситтетическая зависимость для началбного дебита нефти, bbl/d
qi_bopd= (1100*porosity
          +2.2*permeability_md
          +8*net_pay_thickness_ft
          +11*choke_64in
          -900*water_cut
          -250*boi_rb_stb
          +noise
          ).clip(50, None) 

#формирование и сохранение таблицы данных в формате CSV
df= pd.DataFrame({'porosity_frac': porosity,
                 'permeability_md': permeability_md,
                 'net_pay_thickness_ft': net_pay_thickness_ft,
                 'boi_rb_stb': boi_rb_stb,
                 'choke_64in': choke_64in,
                 'water_cut': water_cut,
                 'qi_bopd': qi_bopd},
                 )

df.to_csv

df

,porosity_frac,permeability_md,net_pay_thickness_ft,boi_rb_stb,choke_64in,water_cut,qi_bopd
0,0.234791,65.203086,68.869217,1.450684,55.677351,0.155037,787.591376
1,0.167776,74.409591,97.283361,1.460657,42.938707,0.446241,556.739885
2,0.251720,67.063670,98.612027,1.493827,34.497735,0.528143,468.690882
3,0.219474,89.016380,35.940542,1.492947,28.993445,0.628388,50.000000
4,0.098835,224.095212,76.037815,1.481200,41.052087,0.205359,831.155043
...,...,...,...,...,...,...,...
295,0.260531,37.770694,54.595590,1.340623,51.355508,0.070368,697.541448
296,0.275914,131.907929,19.962287,1.294477,38.066054,0.323850,281.717244
297,0.240405,323.802644,32.719266,1.204074,26.343767,0.298144,683.673047
298,0.235896,35.106852,36.529655,1.267462,51.769784,0.078964,535.165909


In [62]:
df.describe().round(3)

,porosity_frac,permeability_md,net_pay_thickness_ft,boi_rb_stb,choke_64in,water_cut,qi_bopd
count,300.000,300.000,300.000,300.000,300.000,300.000,300.000
mean,0.177,100.463,53.989,1.305,40.020,0.368,392.669
std,0.058,73.722,26.785,0.117,14.082,0.195,308.065
min,0.081,10.043,10.111,1.100,16.051,0.023,50.000
25%,0.125,49.086,31.332,1.201,26.918,0.191,119.440
50%,0.176,83.009,53.261,1.301,39.261,0.357,347.989
75%,0.230,127.282,77.068,1.409,52.449,0.548,595.741
max,0.278,492.945,99.887,1.500,63.841,0.699,1770.276


 Здесь ```qi_bopd``` - целевая переменная которую престоит предсказывать. Все остальные столбцы - признаки скважины

In [ ]:
#Разделение на признак x и целевую переменную y
x= df.drop(columns= 'qi_bopd')
y= df['qi_bopd']

In [69]:
#80% данных для обучения и 20% для тестирования 
X_train,X_test,y_train,y_test= train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    )

In [71]:
X_train.head(5)

,porosity_frac,permeability_md,net_pay_thickness_ft,boi_rb_stb,choke_64in,water_cut
232,0.273246,40.565663,46.716529,1.428492,19.039767,0.671790
59,0.102906,120.984333,22.313915,1.201274,62.602467,0.297006
6,0.232228,67.777579,41.184358,1.155117,43.076567,0.670112
185,0.212332,91.081489,11.713073,1.187543,41.675111,0.148201
173,0.267209,44.641703,98.618458,1.491282,62.017961,0.100134


In [ ]:
#Модель линейной регрессии
model= LinearRegression()
model.fit(X_train, y_train)

predictions= model.predict(X_test)



array([  35.59472408,  457.68568274,    8.43257725,  411.10885255,
       -167.09515957,  857.84907441,   53.32120711,  575.72268442,
        600.80559769,  435.29658289,  217.6869249 ,  137.68081566,
        974.69929158,  426.04227311, -130.99876292,   70.94152384,
        521.05269434,    4.72913061,  354.15626388,  886.84972752,
        505.02534476,  126.49173256,  926.67372738,  570.20453141,
        409.91462697,  670.61943099,  358.09162392,  641.73682387,
        170.95711362,  726.30090087,  602.48444885,  672.21813641,
        360.34279836,  406.20014973,  258.32604329,  516.76174299,
        456.85181789,  737.73266114,  364.60777868,  609.69185426,
        710.76358325,  534.08927921,  285.23818507,  280.48805147,
        922.08907775,  638.29679534,  889.46979333,  -29.46484822,
       -419.82139329,  763.46961238,  487.57694143,   15.90753278,
        208.72252502,  242.64351628,  682.05636239,  704.13218209,
         27.74758659,  367.34590984,  221.79022727,  191.24860

In [81]:
#Оценка качества модели 
mae= mean_absolute_error(y_test, predictions)
r2= r2_score(y_test, predictions)

mae

48.856397518365426